# Into the Rabbit Hull — scaled-down reimplementation

Phase 1: DINOv2 activations → k-means centroids → trained stable SAE.

See `PAPER_NOTES.md` for the full paper extraction and `README.md` for what's scaled down and why. This is the first of four notebooks — see "What's next" at the bottom.

## 0. Setup

In Colab: add an `HF_TOKEN` secret (Runtime → Secrets) with a HuggingFace token that has accepted the `imagenet-1k` dataset terms, then run the cells below.

In [ ]:
import os, sys

# Anchors on markers unique to this repo so `import config` always finds THIS
# project's src/, regardless of how/where Colab's CWD ended up (upload, git
# clone, Drive mount, ...) — sys.path.append("src") alone only works if CWD
# already happens to be the repo root, which Colab doesn't guarantee.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "sae.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "rabbit_hull"),
    "/content/rabbit_hull",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/sae.py). "
        "cd into the rabbit_hull folder first (e.g. %cd /content/rabbit_hull), "
        "then re-run this cell."
    )

print("Repo root:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from config import CONFIG
import utils
import data_utils
import model_utils
import kmeans_utils
import sae as sae_module

print(CONFIG)

## 1. Load ImageNet-1k validation split and pick the sae_train indices

`sae_train` is a random sample from the val split, drawn only from classes *outside*
the 200 curated ImageNet-200 classes reserved for analysis (`src/imagenet200.py`) —
keeps the SAE dictionary general-purpose and keeps the analysis classes completely
unseen during training. See `README.md` for why 200 classes was chosen over fewer
classes with more images each.

**Run the smoke test cell below first.** It loads a single ~480MB validation shard
(not the gated `ILSVRC/imagenet-1k` loader's own script — see `data_utils.py`'s module
docstring for why that matters) and confirms two things cheaply, in under a minute,
before committing to the real ~6-7GB load: (1) it doesn't also pull train data, and
(2) the class-label metadata this code depends on survives the load.

In [ ]:
data_utils.smoke_test()

In [ ]:
dataset = data_utils.load_imagenet_val(CONFIG)

In [ ]:
sae_train_idx = data_utils.make_sae_train_split(dataset, CONFIG)
print(f"sae_train: {len(sae_train_idx)} images")

## 2. Extract & cache DINOv2 activations

In [ ]:
model, processor = model_utils.load_model(CONFIG)

sae_train_images = [dataset[int(i)]["image"].convert("RGB") for i in sae_train_idx]
sae_train_acts = utils.cached(
    Path(CONFIG["cache_dir"]) / "activations" / "sae_train.pt",
    lambda: model_utils.extract_activations(sae_train_images, model, processor, CONFIG),
)
sae_train_acts.shape

## 3. Fit k-means centroids (conv(A) approximation)

In [ ]:
centroids = utils.cached(
    Path(CONFIG["cache_dir"]) / "centroids.pt",
    lambda: kmeans_utils.fit_centroids(sae_train_acts, CONFIG),
)
centroids.shape

## 4. Train the stable SAE

In [ ]:
sae = sae_module.StableSAE(
    centroids, CONFIG["n_atoms"], CONFIG["sparsity_k"], CONFIG["d_model"]
)
history = sae_module.train_sae(sae, sae_train_acts, CONFIG)

In [ ]:
epochs = [h["epoch"] for h in history]
plt.plot(epochs, [h["r2"] for h in history])
plt.xlabel("epoch")
plt.ylabel("reconstruction R\u00b2")
plt.title("SAE reconstruction quality (paper reports >0.88 at full scale)")
plt.show()

## 5. Save the dictionary checkpoint

In [ ]:
sae_module.save_checkpoint(sae, CONFIG)

## What's next

This notebook covers `PAPER_NOTES.md` Sections 2 and 4 (SAE training). Continue with:

1. **`classification_segmentation.ipynb`** — concept importance (§6), classification & segmentation (ADE20K) task recruitment, Elsewhere concepts, border concepts (§7).
2. **`dictionary_geometry.ipynb`** — token-type/footprint concepts (§8), co-activation spectrum, dictionary geometry & baselines (§9-11).
3. **`mrh_analysis.ipynb`** — position-embedding analysis (§12), MRH empirical evidence: k-NN geodesics, Archetypal Analysis, block structure (§14).

Depth estimation (§7's third downstream task) and the DinoVision visualization tool (§16) stay out of scope — see `README.md`'s differences-from-paper table.